# AI Landscape Monitor — Colab Project Builder

**Academic project:** Google, OpenAI, Anthropic, NVIDIA and Apple. This Colab notebook writes a complete Streamlit app, RAG research pipeline, optional AI agent, SEC 10-K collector and scheduled GitHub refresh workflow.

**How to use:** Runtime → Run all. It will create the files in `/content/ai_landscape_dashboard`, install dependencies, and give you a downloadable ZIP. You can also download the project ZIP provided alongside this notebook without running Colab.

No OpenAI API key is necessary to build the dashboard or run the free evidence-only RAG mode. Generative agent mode is optional and billable. Do not put secrets in the notebook or GitHub.

In [ ]:
from pathlib import Path
import os
project = Path('/content/ai_landscape_dashboard')
project.mkdir(exist_ok=True)
for directory in ['.github/workflows', '.streamlit', 'data', 'tests']:
    (project / directory).mkdir(parents=True, exist_ok=True)
os.chdir(project)
print('Working directory:', os.getcwd())

### Generate `requirements.txt`

In [ ]:
%%writefile requirements.txt
streamlit>=1.44,<2
streamlit-autorefresh>=1.0,<2
requests>=2.31,<3
beautifulsoup4>=4.12,<5
feedparser>=6.0,<7
scikit-learn>=1.4,<2
pandas>=2.1,<3
plotly>=5.20,<7
openai>=2,<4


### Generate `research.py`

In [ ]:
%%writefile research.py
"""Verified-source collection, SEC finance data, retrieval, and optional tool-using research agent.
Only official corporate domains are admitted to the research corpus.
"""
from __future__ import annotations

import json
import os
import re
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import urljoin, urlsplit, urlunsplit

import feedparser
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

COMPANIES = ["Google", "OpenAI", "Anthropic", "NVIDIA", "Apple"]
DOMAINS = {
    "Google": ["blog.google", "deepmind.google", "google.com", "cloud.google.com", "abc.xyz"],
    "OpenAI": ["openai.com"],
    "Anthropic": ["anthropic.com"],
    "NVIDIA": ["nvidia.com", "blogs.nvidia.com"],
    "Apple": ["apple.com", "machinelearning.apple.com"],
}
RSS_FEEDS = {
    "Google": ["https://blog.google/technology/ai/rss/", "https://deepmind.google/blog/rss.xml"],
    "OpenAI": ["https://openai.com/news/rss.xml"],
    "Anthropic": [],  # Anthropic does not publish an official news RSS feed; use its newsroom.
    "NVIDIA": ["https://blogs.nvidia.com/feed/"],
    "Apple": ["https://www.apple.com/newsroom/rss-feed.rss"],
}
NEWSROOMS = {
    "Google": "https://blog.google/technology/ai/",
    "OpenAI": "https://openai.com/news/",
    "Anthropic": "https://www.anthropic.com/news",
    "NVIDIA": "https://blogs.nvidia.com/",
    "Apple": "https://www.apple.com/newsroom/",
}
REFERENCE_PAGES = {
    "Google": [("Google AI official updates", "https://blog.google/technology/ai/")],
    "OpenAI": [("OpenAI official news", "https://openai.com/news/")],
    "Anthropic": [("Anthropic official news", "https://www.anthropic.com/news")],
    "NVIDIA": [("NVIDIA official AI and company blog", "https://blogs.nvidia.com/")],
    "Apple": [("Apple Newsroom official updates", "https://www.apple.com/newsroom/")],
}
SEC_CIK = {"Google": "0001652044", "NVIDIA": "0001045810", "Apple": "0000320193"}
SEC_FORMS = ("10-K", "10-Q")
REVENUE_TAGS = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues", "SalesRevenueNet", "RevenueFromContractWithCustomerIncludingAssessedTax",
]
USER_AGENT = "Mozilla/5.0 (compatible; AcademicAILandscapeDashboard/1.0)"
TIMEOUT = 15
KEYWORDS = {
    "Models & products": ["model", "gemini", "claude", "gpt", "apple intelligence", "inference", "platform", "launch"],
    "Partnerships": ["partner", "collaborat", "agreement", "alliance", "joint", "integrat"],
    "Infrastructure & chips": ["gpu", "chip", "data center", "datacenter", "compute", "infrastructure", "accelerat"],
    "Policy & safety": ["safety", "regulat", "privacy", "policy", "security", "responsib"],
    "Financial performance": ["earnings", "revenue", "fiscal", "quarter", "financial", "invest"],
}


def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def normalize_url(url):
    """Remove fragments and tracking query parameters; only use http(s)."""
    try:
        p = urlsplit(str(url).strip())
        if p.scheme not in ("http", "https") or not p.hostname:
            return ""
        return urlunsplit((p.scheme, p.netloc.lower(), p.path or "/", "", ""))
    except ValueError:
        return ""


def verified_domain(url, company):
    """Domain allowlist. No unverified aggregator articles can become sources."""
    try:
        p = urlsplit(url)
        host = (p.hostname or "").lower()
        return p.scheme == "https" and any(host == d or host.endswith("." + d) for d in DOMAINS[company])
    except (ValueError, KeyError):
        return False


def parse_date(value):
    if not value:
        return None
    try:
        s = str(value).strip()
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
            return s
        return datetime.fromisoformat(s.replace("Z", "+00:00")).date().isoformat()
    except (ValueError, TypeError):
        try:
            return parsedate_to_datetime(str(value)).date().isoformat()
        except (ValueError, TypeError, IndexError):
            return None


def strip_text(text, limit=5500):
    return re.sub(r"\s+", " ", BeautifulSoup(str(text or ""), "html.parser").get_text(" ")).strip()[:limit]


def get_session(sec_contact_email=None):
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT, "Accept": "text/html,application/rss+xml,application/xml,application/json"})
    if sec_contact_email:
        session.headers["User-Agent"] = f"Academic AI research dashboard {sec_contact_email}"
    return session


def extract_article(html, fallback_title=""):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.select("script,style,nav,footer,header,form,aside"):
        tag.decompose()
    title = (soup.find("h1") or soup.find("title"))
    title = strip_text(title.get_text(" ") if title else fallback_title, limit=240) or fallback_title
    node = soup.find("article") or soup.find("main") or soup.body or soup
    parts = []
    for item in node.find_all(["p", "h2", "h3", "li"], limit=180):
        t = strip_text(item.get_text(" "), 500)
        if len(t) > 35 and t not in parts:
            parts.append(t)
    body = " ".join(parts)[:5500]
    description = soup.find("meta", attrs={"name": "description"}) or soup.find("meta", attrs={"property": "og:description"})
    if len(body) < 90 and description:
        body = strip_text(description.get("content", ""), limit=5500)
    date_meta = soup.find("meta", attrs={"property": "article:published_time"})
    date_meta = date_meta or soup.find("meta", attrs={"name": "date"})
    date_meta = date_meta or soup.find("time", attrs={"datetime": True})
    date = parse_date((date_meta.get("content") or date_meta.get("datetime")) if date_meta else None)
    return title, body, date


def make_doc(company, url, title, content, date=None, kind="Newsroom article"):
    url = normalize_url(url)
    if not verified_domain(url, company):
        return None
    return {
        "id": f"{company.lower()}-{abs_hash(url)}",
        "company": company, "title": strip_text(title, 240), "url": url,
        "published": parse_date(date), "content": strip_text(content, 5500),
        "source_type": kind, "domain_verified": True,
    }


def abs_hash(s):
    import hashlib
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:12]


def fetch_document(session, company, url, title="", published=None, source_type="Newsroom article"):
    url = normalize_url(url)
    if not verified_domain(url, company):
        return None
    try:
        r = session.get(url, timeout=TIMEOUT, allow_redirects=True)
        r.raise_for_status()
        final_url = normalize_url(r.url)
        if not verified_domain(final_url, company):
            return None
        page_title, body, page_date = extract_article(r.text, title)
        if not body:
            return None
        return make_doc(company, final_url, title or page_title, body, published or page_date, source_type)
    except requests.RequestException:
        return None


def feed_candidates(session, company):
    results, warnings = [], []
    for feed_url in RSS_FEEDS[company]:
        try:
            r = session.get(feed_url, timeout=TIMEOUT)
            r.raise_for_status()
            parsed = feedparser.parse(r.content)
            if not parsed.entries:
                warnings.append(f"No entries in official feed: {feed_url}")
            for e in parsed.entries[:18]:
                link = normalize_url(e.get("link", ""))
                if verified_domain(link, company):
                    results.append((link, e.get("title", ""), e.get("published") or e.get("updated")))
        except requests.RequestException as e:
            warnings.append(f"Official feed unavailable: {feed_url} ({type(e).__name__})")
    return results, warnings


def newsroom_candidates(session, company):
    url = NEWSROOMS[company]
    try:
        r = session.get(url, timeout=TIMEOUT)
        r.raise_for_status()
        if not verified_domain(normalize_url(r.url), company):
            return []
        soup = BeautifulSoup(r.text, "html.parser")
        result, seen = [], set()
        for a in soup.find_all("a", href=True):
            link = normalize_url(urljoin(r.url, a["href"]))
            label = strip_text(a.get_text(" "), 240)
            path = urlsplit(link).path.rstrip("/").lower()
            if not verified_domain(link, company) or link in seen or len(label) < 14:
                continue
            if company == "Anthropic":
                valid = "/news/" in path or "/research/" in path
            elif company == "Apple":
                valid = "/newsroom/20" in path
            elif company == "Google":
                valid = "/technology/ai/" in path or "/blog/" in path
            elif company == "NVIDIA":
                valid = path.count("/") >= 2
            else:
                valid = "/index/" in path or "/news/" in path
            if valid and len(path) > 15 and link != normalize_url(url):
                seen.add(link)
                result.append((link, label, None))
            if len(result) >= 22:
                break
        return result
    except requests.RequestException:
        return []


def collect_news(max_per_company=8):
    """Returns verified official article text, with failure telemetry; no invented news."""
    session = get_session()
    docs, warnings, sources = [], [], {}
    for company in COMPANIES:
        candidates, feed_warnings = feed_candidates(session, company)
        warnings.extend(feed_warnings)
        candidates.extend(newsroom_candidates(session, company))
        seen = set()
        company_docs = []
        for url, title, date in candidates:
            if url in seen:
                continue
            seen.add(url)
            doc = fetch_document(session, company, url, title, date)
            if doc and len(doc["content"]) >= 90:
                company_docs.append(doc)
            if len(company_docs) >= max_per_company:
                break
        # Reliable primary-source context even if a feed is unavailable.
        if not company_docs:
            for title, url in REFERENCE_PAGES[company]:
                doc = fetch_document(session, company, url, title, source_type="Official reference page")
                if doc:
                    company_docs.append(doc)
        if not company_docs:
            warnings.append(f"No verified content could be fetched for {company}; no results fabricated.")
        sources[company] = len(company_docs)
        docs.extend(company_docs)
    docs.sort(key=lambda d: d.get("published") or "0000-00-00", reverse=True)
    return {"collected_at": now_utc(), "documents": docs, "source_counts": sources, "warnings": warnings}


def fetch_sec_annual_revenue(company, contact_email):
    """USD annual revenue from 10-K Company Facts (not stock market cap)."""
    if company not in SEC_CIK:
        return [], "Private / no comparable public-company 10-K revenue series"
    if not contact_email or not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", contact_email):
        return [], "Add SEC_CONTACT_EMAIL in Streamlit secrets to access SEC data."
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{SEC_CIK[company]}.json"
    try:
        r = get_session(contact_email).get(url, timeout=25)
        r.raise_for_status()
        data = r.json()
        us_gaap = data.get("facts", {}).get("us-gaap", {})
        entries = []
        for tag in REVENUE_TAGS:
            values = us_gaap.get(tag, {}).get("units", {}).get("USD", [])
            for item in values:
                try:
                    duration = (datetime.fromisoformat(item["end"]) - datetime.fromisoformat(item["start"])).days
                    if item.get("form") == "10-K" and item.get("fp") == "FY" and 320 <= duration <= 400 and item.get("val") is not None:
                        entries.append({"company": company, "end": item["end"], "fy": int(item["fy"]),
                                        "value_usd": int(item["val"]), "filed": item.get("filed", ""),
                                        "accession": item.get("accn", ""), "tag": tag,
                                        "url": url, "source": "SEC EDGAR XBRL Company Facts (10-K)"})
                except (KeyError, ValueError, TypeError):
                    pass
            if entries:
                break
        # Later-filed facts can restate prior years. Keep newest for each fiscal period end.
        latest = {}
        for e in entries:
            key = e["end"]
            if key not in latest or e["filed"] > latest[key]["filed"]:
                latest[key] = e
        return sorted(latest.values(), key=lambda e: e["end"])[-5:], None
    except (requests.RequestException, ValueError) as e:
        return [], f"SEC request failed for {company}: {type(e).__name__}"


def collect_financials(contact_email):
    rows, warnings = [], []
    for company in SEC_CIK:
        company_rows, warning = fetch_sec_annual_revenue(company, contact_email)
        rows.extend(company_rows)
        if warning:
            warnings.append(warning)
    return {"collected_at": now_utc(), "rows": rows, "warnings": warnings}


def topic_matches(document):
    text = (document.get("title", "") + " " + document.get("content", "")).lower()
    return [topic for topic, terms in KEYWORDS.items() if any(term in text for term in terms)]


def retrieve(query, documents, companies=None, limit=5):
    """TF-IDF retriever: lexical RAG retrieval over explicitly verified primary sources."""
    docs = [d for d in documents if d.get("domain_verified") and (not companies or d.get("company") in companies)]
    if not docs or not str(query).strip():
        return []
    corpus = [(d["title"] + " " + d.get("content", ""))[:6000] for d in docs]
    try:
        vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=15000)
        matrix = vectorizer.fit_transform(corpus + [query])
        scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
        order = sorted(range(len(scores)), key=lambda i: float(scores[i]), reverse=True)
        return [{**docs[i], "relevance": round(float(scores[i]), 4)} for i in order[:limit] if scores[i] > 0]
    except ValueError:
        return []


def make_evidence_answer(question, documents):
    matches = retrieve(question, documents, limit=5)
    if not matches:
        return "No matching passages found in the currently verified source corpus. Refresh or broaden the query.", []
    snippets = []
    for i, d in enumerate(matches[:4], 1):
        sentence_candidates = re.split(r"(?<=[.!?])\s+", d.get("content", ""))
        terms = set(re.findall(r"\b[a-z]{4,}\b", question.lower()))
        sentences = sorted(sentence_candidates, key=lambda s: sum(t in s.lower() for t in terms), reverse=True)
        excerpt = next((s.strip() for s in sentences if len(s) > 35), d["content"][:290])[:340]
        snippets.append(f"[{i}] {d['company']} — {d['title']}: {excerpt}")
    answer = "EXTRACTIVE EVIDENCE (not an AI-generated conclusion):\n\n" + "\n\n".join(snippets)
    return answer, matches[:4]


def run_research_agent(question, documents, financial_rows, api_key, model="gpt-4.1-mini"):
    """True model-driven tool selection via OpenAI Responses function calls.
    A tool must be called first. Models are forced to ground conclusions in returned records.
    """
    from openai import OpenAI
    client = OpenAI(api_key=api_key, timeout=35, max_retries=1)
    tools = [
        {"type": "function", "name": "search_official_sources",
         "description": "Search verified first-party company announcements and extract evidence with URLs.",
         "parameters": {"type": "object", "properties": {"query": {"type": "string"},
                         "company": {"type": "string", "enum": COMPANIES + ["All"]}},
                        "required": ["query", "company"], "additionalProperties": False}},
        {"type": "function", "name": "get_annual_revenue",
         "description": "Get available SEC 10-K annual revenue in USD for the three public firms; no private-company estimates.",
         "parameters": {"type": "object", "properties": {"company": {"type": "string", "enum": COMPANIES + ["All"]}},
                        "required": ["company"], "additionalProperties": False}},
    ]
    instructions = (
        "You are a skeptical AI industry research agent for investors and strategists. "
        "Call the supplied source tools and answer ONLY with grounded facts returned by them. "
        "Link each substantive claim to the actual source URL in returned evidence. "
        "Distinguish issuer claims from independent verification; these are FIRST-PARTY SOURCES. "
        "Company announcements are not independent proof of outcomes. "
        "No unsupported market share, stock prices, token costs, projections, or private revenue. "
        "Explain missing/old data and fiscal-year differences. Keep answers under 450 words."
    )
    input_items = [{"role": "user", "content": question}]
    trace = []
    for iteration in range(4):
        response = client.responses.create(model=model, instructions=instructions, input=input_items,
                                           tools=tools, tool_choice="required" if iteration == 0 else "auto",
                                           max_output_tokens=900, store=False)
        input_items += response.output
        calls = [x for x in response.output if x.type == "function_call"]
        if not calls:
            return response.output_text or "No response was generated.", trace
        for call in calls:
            args = json.loads(call.arguments)
            if call.name == "search_official_sources":
                target = args.get("company", "All")
                matches = retrieve(args.get("query", question), documents,
                                   companies=None if target == "All" else [target], limit=6)
                payload = [{k: d.get(k) for k in ("company", "title", "published", "url", "relevance")}
                           | {"excerpt": d.get("content", "")[:1100]} for d in matches]
            elif call.name == "get_annual_revenue":
                target = args.get("company", "All")
                payload = [r for r in financial_rows if target == "All" or r["company"] == target]
            else:
                payload = {"error": "Unknown tool"}
            trace.append({"tool": call.name, "arguments": args, "results": len(payload) if isinstance(payload, list) else 0})
            input_items.append({"type": "function_call_output", "call_id": call.call_id,
                                "output": json.dumps(payload, ensure_ascii=False)[:17000]})
    return "Tool-call limit reached; see the evidence and tool trace. No unsupported answer generated.", trace


def load_snapshot(path="data/latest.json"):
    p = Path(path)
    if not p.exists():
        return None
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
        if not isinstance(data.get("news", {}).get("documents"), list):
            return None
        return data
    except (OSError, ValueError, TypeError):
        return None


def write_snapshot(path="data/latest.json", sec_contact_email=""):
    data = {"news": collect_news(), "financials": collect_financials(sec_contact_email), "generated_at": now_utc()}
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    return data


### Generate `app.py`

In [ ]:
%%writefile app.py
"""AI Landscape Monitor: verified first-party research, SEC financials, RAG and tool-using agent."""
from __future__ import annotations

import os
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
import plotly.express as px
import streamlit as st
from streamlit_autorefresh import st_autorefresh

from research import (
    COMPANIES, KEYWORDS, NEWSROOMS, SEC_CIK, collect_financials, collect_news,
    load_snapshot, make_evidence_answer, retrieve, run_research_agent, topic_matches,
)

st.set_page_config(page_title="AI Landscape | Research Dashboard", page_icon="◈", layout="wide")
st_autorefresh(interval=3_600_000, key="hourly_background_refresh")

st.markdown("""
<style>
.block-container {padding-top: 2.0rem; max-width: 1380px;}
.hero {padding: 1.45rem 1.7rem; border: 1px solid #205268; border-radius: 17px;
       background: linear-gradient(110deg,#0e3042,#112538 60%,#082b34); margin-bottom: 1.2rem;}
.hero h1 {font-size: 2.2rem; color: #f3fbff; margin: 0.15rem 0;}
.hero p {color: #b6d5dd; margin: 0.3rem 0;}
.eyebrow {font-size:.78rem; letter-spacing:.15em; color:#5de2ce; font-weight:700;}
.source-card {background:#142a3a;border:1px solid #26485b;border-radius:13px;padding:1rem 1.15rem;margin:.5rem 0;}
.source-card h4 {margin:0 0 .22rem 0;color:#f3fbff;}
.source-card p {margin: .2rem 0; color:#b6cbd7;font-size:.87rem;}
.smallnote {color:#b5c8d7;font-size:.89rem;}
</style>
""", unsafe_allow_html=True)


def secret(key, default=""):
    value = os.environ.get(key, default)
    try:
        value = st.secrets.get(key, value)
    except Exception:
        pass
    return str(value or "")


def age_hours(ts):
    try:
        dt = datetime.fromisoformat(ts.replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return (datetime.now(timezone.utc) - dt).total_seconds() / 3600
    except (ValueError, AttributeError):
        return 1e6


@st.cache_data(ttl=3600, show_spinner="Checking official company sources…")
def get_data(force_live=False, sec_contact_email=""):
    snapshot = load_snapshot()
    if snapshot and not force_live and age_hours(snapshot.get("generated_at", "")) < 12:
        return snapshot, "Scheduled verified-source snapshot"
    news = collect_news(max_per_company=8)
    financials = collect_financials(sec_contact_email)
    if not news["documents"] and snapshot:
        return snapshot, "Saved verified-source snapshot (live sources temporarily unavailable)"
    return {"generated_at": news["collected_at"], "news": news, "financials": financials}, "Live first-party collection"


with st.sidebar:
    st.markdown("### ◈ Research settings")
    companies = st.multiselect("Companies", COMPANIES, default=COMPANIES)
    topic = st.selectbox("Topic", ["All topics"] + list(KEYWORDS))
    search_text = st.text_input("Article search", placeholder="GPU, partnership, model launch…")
    st.markdown("---")
    st.caption("Auto-refresh is hourly while a viewer has the app open. The included GitHub Actions workflow also attempts a source refresh every 6 hours.")
    if st.button("↻ Refresh from official websites", use_container_width=True):
        st.cache_data.clear()
        st.session_state["force_live_once"] = True
        st.rerun()
    st.caption("**Data policy:** first-party corporate announcements and SEC filings only. No invented stock prices, market shares or private-company revenues.")
    st.markdown("### Official newsrooms")
    for company in COMPANIES:
        st.markdown(f"[{company}]({NEWSROOMS[company]})")

force_live = st.session_state.pop("force_live_once", False)
sec_email = secret("SEC_CONTACT_EMAIL")
try:
    bundle, data_mode = get_data(force_live=force_live, sec_contact_email=sec_email)
except Exception as err:
    st.error("Unable to load the research corpus. Try refreshing. " + type(err).__name__)
    st.stop()

news = bundle.get("news", {})
financials = bundle.get("financials", {})
all_docs = news.get("documents", [])
financial_rows = financials.get("rows", [])
shown = [d for d in all_docs if d["company"] in companies]
if topic != "All topics":
    shown = [d for d in shown if topic in topic_matches(d)]
if search_text:
    q = search_text.lower()
    shown = [d for d in shown if q in (d.get("title", "") + " " + d.get("content", "")).lower()]

def display_date(date_str):
    return date_str or "Date not disclosed"


def source_card(doc):
    st.markdown(f"<div class='source-card'><div class='smallnote'>{doc['company']} · "
                f"{display_date(doc.get('published'))} · {doc.get('source_type', '')}</div>"
                f"<h4>{html_escape(doc['title'])}</h4>"
                f"<p>{html_escape(doc.get('content', '')[:280])}…</p></div>",
                unsafe_allow_html=True)
    st.link_button("Read original verified source ↗", doc["url"], use_container_width=False)


def html_escape(text):
    import html
    return html.escape(str(text), quote=True)


st.markdown("""
<div class="hero">
  <div class="eyebrow">AI INDUSTRY INTELLIGENCE · RESEARCH DASHBOARD</div>
  <h1>AI Landscape Monitor</h1>
  <p>Google · OpenAI · Anthropic · NVIDIA · Apple</p>
  <p>Official announcements, SEC filings, source-grounded retrieval and an optional research agent.</p>
</div>
""", unsafe_allow_html=True)

st.caption(f"**Collection mode:** {data_mode}  |  **Snapshot generated (UTC):** {bundle.get('generated_at', 'Unknown')}  |  "
           "**Verification:** URL host must match an approved official corporate domain")

if not all_docs:
    st.warning("No verified articles were accessible. The dashboard will not fabricate news. "
               "Check deployment internet access, the official source links, or run the scheduled refresh workflow.")

k1, k2, k3, k4 = st.columns(4)
k1.metric("Verified source pages", len(all_docs))
k2.metric("Companies monitored", sum(bool(news.get("source_counts", {}).get(c, 0)) for c in COMPANIES))
k3.metric("SEC annual observations", len(financial_rows))
k4.metric("Source categories", len(KEYWORDS))

pages = st.tabs(["Overview", "Company comparisons", "Financial evidence", "Research agent + RAG", "Sources & methodology"])

with pages[0]:
    st.subheader("Overview for business leaders and market researchers")
    st.info("**Interpretation:** This dashboard measures the volume and content of *official company announcements*, "
            "not industry market share, business outcomes, or independent claims verification.")
    if shown:
        df = pd.DataFrame([{"Company": d["company"], "Source": d["title"], "Date": d.get("published"),
                            "Topics": ", ".join(topic_matches(d)) or "Other"} for d in shown])
        left, right = st.columns([1, 1.25])
        with left:
            tally = df.groupby("Company").size().reset_index(name="Verified pages")
            fig = px.bar(tally, x="Company", y="Verified pages", color="Company",
                         title="Official sources in current corpus", text="Verified pages")
            fig.update_layout(showlegend=False, margin=dict(l=5, r=5, t=55, b=5))
            st.plotly_chart(fig, use_container_width=True)
        with right:
            topics = Counter(tag for d in shown for tag in topic_matches(d))
            if topics:
                rows = [{"Topic": t, "Documents mentioning it": n} for t, n in topics.items()]
                fig = px.bar(pd.DataFrame(rows), x="Documents mentioning it", y="Topic", orientation="h",
                             title="Topic mention signals in official pages")
                fig.update_layout(margin=dict(l=5, r=5, t=55, b=5))
                st.plotly_chart(fig, use_container_width=True)
        dated = [d for d in shown if d.get("published")]
        if dated:
            dated_df = pd.DataFrame([{"Month": d["published"][:7], "Company": d["company"], "Count": 1} for d in dated])
            by_month = dated_df.groupby(["Month", "Company"], as_index=False)["Count"].sum()
            fig = px.line(by_month, x="Month", y="Count", color="Company", markers=True,
                          title="Official publications by month (only sources with dates)")
            st.plotly_chart(fig, use_container_width=True)
        st.subheader("Recent source-grounded developments")
        for d in shown[:6]:
            with st.expander(f"{d['company']} · {d['title']}", expanded=False):
                st.write(d.get("content", "")[:1400])
                st.caption("Published: " + display_date(d.get("published")))
                st.link_button("View original", d["url"])
    else:
        st.info("No sources match the selected filters. Clear the sidebar filters or refresh.")
    st.markdown("#### Decisions this report supports")
    st.write("**Strategists:** track product and partnership announcements. **Investors:** inspect SEC-reported historical revenue "
             "for the public firms only. **Researchers:** trace claims back to official sources and ask citation-grounded questions.")

with pages[1]:
    st.subheader("Compare competitive activity")
    if shown:
        rows = []
        for d in shown:
            for t in topic_matches(d):
                rows.append({"Company": d["company"], "Topic": t, "Source count": 1})
        if rows:
            grouped = pd.DataFrame(rows).groupby(["Company", "Topic"], as_index=False)["Source count"].sum()
            fig = px.bar(grouped, x="Company", y="Source count", color="Topic", barmode="group",
                         title="Verified corporate source coverage by topic")
            st.plotly_chart(fig, use_container_width=True)
        st.caption("Counts depend on the limited collected corpus and source publishing volume; "
                   "they do not measure market share or provide an exhaustive industry ranking.")
    cols = st.columns(2)
    for i, company in enumerate(COMPANIES):
        with cols[i % 2]:
            company_docs = [d for d in shown if d["company"] == company]
            with st.container(border=True):
                st.markdown(f"#### {company}")
                st.caption(f"{len(company_docs)} matching verified records")
                if company_docs:
                    d = company_docs[0]
                    st.write(f"**Latest in selected corpus:** {d['title']}")
                    st.caption(display_date(d.get("published")))
                    st.link_button("Read primary source", d["url"], key=f"co-{company}")
                else:
                    st.write("No matching records; source may be unavailable or filtered out.")

with pages[2]:
    st.subheader("SEC filings: audited-source financial evidence")
    st.write("These figures come from SEC EDGAR structured XBRL annual-revenue disclosures (Form 10-K). "
             "Google is represented by Alphabet. OpenAI and Anthropic are private companies and have "
             "no directly comparable 10-K series here.")
    if financial_rows:
        fdf = pd.DataFrame(financial_rows).copy()
        fdf["Revenue (USD billions)"] = fdf["value_usd"] / 1e9
        fdf["Fiscal period end"] = fdf["end"]
        fig = px.line(fdf.sort_values("Fiscal period end"), x="Fiscal period end", y="Revenue (USD billions)",
                      color="company", markers=True, title="Annual revenue in SEC 10-K filings (fiscal years differ)")
        fig.update_layout(legend_title_text="Company")
        st.plotly_chart(fig, use_container_width=True)
        table = fdf[["company", "Fiscal period end", "Revenue (USD billions)", "filed", "url"]].rename(
            columns={"company": "Company", "filed": "SEC filed on", "url": "Primary SEC source"})
        st.dataframe(table, hide_index=True, use_container_width=True,
                     column_config={"Primary SEC source": st.column_config.LinkColumn()})
        st.warning("**Comparability:** fiscal calendars differ, SEC facts may be restated, and the chart is not a stock-price "
                   "or profitability comparison. Values may reflect issuer-specific XBRL tags.")
    else:
        st.info("SEC annual data is not populated. Set `SEC_CONTACT_EMAIL` in Streamlit **Secrets**, "
                "or in GitHub Actions repository secrets, then refresh. Use your real contact email as SEC requires.")
    st.caption("Official SEC API: https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json")
    for warning in financials.get("warnings", []):
        st.caption("Source status: " + warning)

with pages[3]:
    st.subheader("Source-grounded research assistant")
    st.markdown("**RAG pipeline:** verified-source ingestion → article text → TF-IDF retrieval → "
                "evidence display → optional LLM tool-calling agent → URL-cited answer.")
    st.caption("The retrieval engine works without a paid key. Generative answers and tool-calling "
               "require an OpenAI API key with API billing enabled, separate from ChatGPT subscription.")
    question = st.text_input("Ask an evidence-based research question", value="How are Google and OpenAI competing in AI models?", key="ask")
    answer_mode = st.radio("Mode", ["Free evidence retrieval", "OpenAI tool-using research agent"], horizontal=True)
    model = secret("OPENAI_MODEL", "gpt-4.1-mini")
    api_key = secret("OPENAI_API_KEY")
    if answer_mode.startswith("OpenAI") and not api_key:
        st.info("To use the actual model-driven agent, add OPENAI_API_KEY to the app's Streamlit Secrets. "
                "Never paste your API key in a public GitHub file.")
    if st.button("Run research", type="primary", key="run_research", disabled=not bool(question.strip())):
        with st.spinner("Retrieving verified evidence…"):
            if answer_mode.startswith("Free"):
                answer, matches = make_evidence_answer(question, shown or all_docs)
                trace = [{"tool": "retrieve_tfidf", "arguments": {"query": question}, "results": len(matches)}]
            elif not api_key:
                answer, matches = "No API key configured; showing evidence-only retrieval instead.", retrieve(question, shown or all_docs)
                trace = []
            else:
                try:
                    answer, trace = run_research_agent(question, shown or all_docs, financial_rows, api_key, model=model)
                    matches = retrieve(question, shown or all_docs)
                except Exception as exc:
                    answer, matches = make_evidence_answer(question, shown or all_docs)
                    answer = (f"The OpenAI agent was unavailable ({type(exc).__name__}). "
                              "Showing extractive evidence instead.\n\n" + answer)
                    trace = []
            st.session_state["research_output"] = (answer, matches, trace)
    if "research_output" in st.session_state:
        answer, matches, trace = st.session_state["research_output"]
        st.markdown("#### Answer / evidence")
        st.text(answer) if answer_mode.startswith("Free") else st.write(answer)
        st.markdown("#### Retrieved primary-source citations")
        for i, d in enumerate(matches, 1):
            st.markdown(f"**[{i}] [{d['title']}]({d['url']})** — {d['company']} · "
                        f"{display_date(d.get('published'))} · similarity {d.get('relevance', 0):.3f}")
        with st.expander("Show agent / retrieval audit trace"):
            st.json(trace)
    st.markdown("#### Questions worth testing")
    st.write("• Which firms announced new AI products recently?  • What partnerships are mentioned in official releases?  "
             "• What can we verify about NVIDIA's infrastructure role?  • How do Alphabet, Apple and NVIDIA annual revenues differ?")

with pages[4]:
    st.subheader("Trust, citations and reproducibility")
    st.markdown("**Scope:** Google (Alphabet), OpenAI, Anthropic, NVIDIA and Apple. "
                "Only approved official corporate domains are indexed; SEC EDGAR provides public-company financial figures. "
                "This verifies *source origin*, not that corporate marketing claims are independently true.")
    st.markdown("**Update mechanism:** GitHub Actions scheduled refresh every 6 hours (subject to GitHub scheduling, "
                "permissions, and inactivity limits); live checks when the dashboard starts and hourly reruns while open. "
                "Source failures are surfaced rather than replaced with fabricated observations.")
    st.markdown("**RAG:** scikit-learn TF-IDF over collected article text. **Agent:** OpenAI Responses API function "
                "calling with source-search and SEC-revenue tools; outputs require citations to URLs. "
                "**Limits:** incomplete media coverage, newsrooms may change markup, corporate-source bias, "
                "and missing private-company data. Financial figures reflect fiscal-year reporting calendars.")
    st.markdown("#### Full source register")
    if shown:
        register = pd.DataFrame([{
            "Company": d["company"], "Title": d["title"], "Published": d.get("published") or "Unspecified",
            "Type": d.get("source_type", ""), "Verified": "Official domain", "URL": d["url"]
        } for d in shown])
        st.dataframe(register, hide_index=True, use_container_width=True, column_config={"URL": st.column_config.LinkColumn()})
        st.download_button("Download current source register (CSV)", register.to_csv(index=False).encode("utf-8"),
                           file_name="verified_ai_sources.csv", mime="text/csv")
    st.markdown("#### Collector health")
    for company in COMPANIES:
        st.write(f"**{company}:** {news.get('source_counts', {}).get(company, 0)} verified article/page records")
    if news.get("warnings"):
        with st.expander("Collection warnings and feed status"):
            for warning in news["warnings"]:
                st.caption(warning)
    st.markdown("#### Research ethics")
    st.caption("AI summaries can be wrong; cross-check cited articles. No claims of independent validation, "
               "exhaustive indexing, or investment advice. For a class presentation, show primary-source links "
               "and the tool audit trace.")

st.divider()
st.caption("Developed as a business analytics research demonstration · Official-source attribution · "
           "Every graph reflects retrieved records, not simulated data")


### Generate `refresh_data.py`

In [ ]:
%%writefile refresh_data.py
"""GitHub Actions scheduled refresh. Run `python refresh_data.py` manually as needed."""
import os
from research import write_snapshot

if __name__ == "__main__":
    data = write_snapshot(sec_contact_email=os.getenv("SEC_CONTACT_EMAIL", ""))
    print(f"Snapshot refreshed: {len(data['news']['documents'])} official documents, "
          f"{len(data['financials']['rows'])} SEC annual revenue observations")
    for warning in data["news"]["warnings"] + data["financials"]["warnings"]:
        print("NOTE:", warning)


### Generate `.streamlit/config.toml`

In [ ]:
%%writefile .streamlit/config.toml
[theme]
primaryColor = "#14B8A6"
backgroundColor = "#07131F"
secondaryBackgroundColor = "#122536"
textColor = "#E7F0F9"
font = "sans serif"


### Generate `.gitignore`

In [ ]:
%%writefile .gitignore
__pycache__/
.pytest_cache/
.env
.streamlit/secrets.toml
.ipynb_checkpoints/
*.pyc


### Generate `.github/workflows/refresh.yml`

In [ ]:
%%writefile .github/workflows/refresh.yml
name: Refresh verified AI industry research

on:
  schedule:
    - cron: '17 */6 * * *'
  workflow_dispatch:

permissions:
  contents: write

jobs:
  refresh:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: pip
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Fetch verified sources and SEC financials
        env:
          SEC_CONTACT_EMAIL: ${{ secrets.SEC_CONTACT_EMAIL }}
        run: python refresh_data.py
      - name: Commit changed snapshot
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "41898282+github-actions[bot]@users.noreply.github.com"
          git add data/latest.json
          if ! git diff --cached --quiet; then
            git commit -m "Auto-refresh verified AI news and SEC figures"
            git push
          fi


### Generate `README.md`

In [ ]:
%%writefile README.md
# AI Landscape Monitor — MIS research assignment

A deployable **Streamlit business intelligence dashboard** covering **Google, OpenAI, Anthropic, NVIDIA and Apple**. The app gathers only verified official corporate-domain source content, can retrieve SEC annual revenue from EDGAR for the three publicly traded companies, supports **TF-IDF retrieval-augmented research**, and (with an optional paid API key) runs an **OpenAI Responses API tool-using research agent**.

## What it demonstrates

- **Acquisition:** corporate RSS feeds, official newsroom fallback HTML, and SEC XBRL Company Facts API.
- **Source checking:** domain allowlist after redirects, source URLs, dated articles when published dates are available; warnings on failed sources.
- **Analysis:** company/topic coverage, source-level trends, historical SEC annual revenue with explicit comparability warnings.
- **RAG:** real TF-IDF retrieval over the collected article text. No API key required to retrieve/cite passages.
- **Agent:** genuine LLM-selected tools (`search_official_sources`, `get_annual_revenue`) with a visible tool trace. Requires an OpenAI API key and API billing.
- **Refresh:** a GitHub Actions workflow attempts source ingestion every 6 hours, plus Streamlit hourly in-session reload and manual refresh. GitHub may delay/discontinue unattended scheduled jobs on inactive repositories.

## Start in Colab

1. Open the included `AI_Landscape_Colab_Setup.ipynb` in Google Colab.
2. Run the cells in order. This generates the Python files, checks code validity and creates a ZIP for GitHub.
3. No paid API key is needed for the dashboard, source retrieval, or official article search. Optional OpenAI agent calls incur usage costs.

## Deploy with Streamlit Community Cloud

1. Create a **new GitHub repository**, e.g. `ai-landscape-dashboard`.
2. Upload the **contents** of this folder to its root: `app.py`, `research.py`, `requirements.txt`, `refresh_data.py`, `.streamlit/config.toml`, `data/`, and `.github/workflows/refresh.yml`.
3. Go to https://share.streamlit.io/ → **Create app** → **Yup, I have an app**.
4. Choose your GitHub repository, branch `main`, and entrypoint `app.py`.
5. Click **Deploy** and share the `*.streamlit.app` URL.
6. For SEC annual revenue, add your genuine contact email as a GitHub Actions repository secret called `SEC_CONTACT_EMAIL` and also as a Streamlit secret. This respects SEC automated access identification. Leave blank if you don't need SEC data.
7. For the optional agent, add `OPENAI_API_KEY` to Streamlit **App settings → Secrets**. Never commit API keys to GitHub. Optional `OPENAI_MODEL = "gpt-4.1-mini"`.
8. In GitHub, open **Actions** → **Refresh verified AI industry research** → **Run workflow** to populate `data/latest.json`, then check the report for source coverage and warnings. If GitHub Actions asks you to enable workflows, do so. Repository workflow permissions must permit writing contents for snapshots to be committed.

Example Streamlit secrets (paste into Streamlit's secure secrets settings, not a GitHub file):

```toml
SEC_CONTACT_EMAIL = "your-real-email@example.com"
# Optional, uses paid OpenAI API credits:
# OPENAI_API_KEY = "sk-..."
# OPENAI_MODEL = "gpt-4.1-mini"
```

## Architecture

```text
Official RSS / HTML newsrooms ----> verify official URL + full page ----> article corpus
SEC Company Facts API ----------> 10-K annual revenue ----> historical chart
article corpus ------------------> TF-IDF retrieval -----> cited extracts (free)
article corpus + revenue -------> OpenAI function tools --> sourced answer + trace (paid optional)
GitHub Actions (6h cron) -------> data/latest.json ------> deployed Streamlit app
```

## Trust and known limitations

- Source provenance is verified through an allowlisted official host, **not independent fact-checking**. Corporate statements can reflect marketing bias.
- Official newsrooms may change their RSS endpoints/HTML structure. Collector warnings report missing data instead of inserting fake results.
- Article dates may be absent; the app labels them as undisclosed and omits them from time series.
- News coverage is not exhaustive. Announcement counts are not product adoption, revenue, market share, or return on investment.
- Only Google/Alphabet, Apple and NVIDIA have SEC filings here. The report deliberately excludes unsupported private-company financial estimates and stock prices.
- Historical SEC 10-K annual revenues reflect non-identical fiscal calendars and sometimes restated disclosures.
- Auto-refresh while the Streamlit page is open differs from unattended server-side cron. The included GitHub Actions job provides the latter, subject to GitHub service restrictions.
- OpenAI API usage is separate from any ChatGPT subscription. Leave the key unset to keep the app's AI retrieval free.

## Presentation suggested sequence (5 minutes)

- 0:00–0:45: stakeholder question and five-company scope.
- 0:45–1:45: show official-source comparisons and a linked announcement.
- 1:45–2:40: compare fiscal revenue across the public firms and explain private-data gaps.
- 2:40–4:10: run an RAG query, show retrieved sources, optionally run the agent and open the tool trace.
- 4:10–5:00: show source verification/auto-refresh limitations and business implications.

## Tests

`python -m pytest -q` (install `pytest` if not installed).


### Generate `tests/test_research.py`

In [ ]:
%%writefile tests/test_research.py
from research import make_doc, normalize_url, parse_date, retrieve, topic_matches, verified_domain, extract_article


def test_domain_validation_blocks_spoofs():
    assert verified_domain("https://openai.com/news/post", "OpenAI")
    assert not verified_domain("https://openai.com.evil.example/news", "OpenAI")
    assert not verified_domain("https://news.google.com/rss/x", "OpenAI")
    assert not verified_domain("http://openai.com/news/post", "OpenAI")


def test_normalize_and_dates():
    assert normalize_url("https://openai.com/news/?utm_source=x#foo") == "https://openai.com/news/"
    assert parse_date("2026-10-20T14:00:00Z") == "2026-10-20"


def test_extractor_and_retrieval():
    html = '<html><head><meta name="date" content="2026-10-20"></head><body><article>' \
           '<h1>A verified launch</h1><p>Our company announced a new model for scientific research and coding.</p>' \
           '</article></body></html>'
    title, body, date = extract_article(html)
    assert date == "2026-10-20"
    doc = make_doc("OpenAI", "https://openai.com/news/some-page", title, body, date)
    assert doc and "Models & products" in topic_matches(doc)
    hits = retrieve("scientific model coding research", [doc])
    assert len(hits) == 1 and hits[0]["url"].startswith("https://openai.com")


### Generate `data/.gitkeep`

In [ ]:
%%writefile data/.gitkeep



### Install and verify

This uses Colab's package installer. Run the test cell afterwards. Full source downloading needs internet and may take a little while.

In [ ]:
!pip -q install -r requirements.txt pytest

In [ ]:
!python -m pytest -q tests/

### Optional: collect the first live dataset

The project automatically downloads verified live official sources when deployed. To create `data/latest.json` now, run this optional cell. **SEC annual revenue requires your real contact email** due to SEC access policy. Never publish it in a notebook. Instead, configure `SEC_CONTACT_EMAIL` using Streamlit/GitHub secrets later.

In [ ]:
# Uncomment the following line to collect official company news now:
# !python refresh_data.py
print("Live collection is optional here; the deployed app can collect when first opened.")

### Package the application for GitHub

Run the cell, then use the browser download. **Extract the ZIP** before uploading the folder's contents into your GitHub repository. GitHub and Streamlit do not execute the ZIP directly.

Go to https://github.com/new → create repository → upload the extracted files (including `.github` and `.streamlit`). Then https://share.streamlit.io → **Create app** → select GitHub repo → entrypoint `app.py`. Check the included `README.md` for Secrets and scheduled refresh steps.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path('/content/AI_Landscape_Streamlit_Project.zip')
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as z:
    for f in project.rglob('*'):
        if f.is_file() and '__pycache__' not in f.parts and '.pytest_cache' not in f.parts:
            z.write(f, f.relative_to(project))
print('Project ZIP ready:', archive)
files.download(str(archive))